# SCOPE $\xi(r)$ convergence

Demonstrates that the **SCOPE sub-volume correction** recovers the full-box real-space
two-point correlation function $\xi(r)$ from a fraction of the simulation realisations.

**Analyses:**
1. **Quick-look** — 4-panel convergence figure for a single config/redshift.
2. **Headline** — naïve vs SCOPE corrected side-by-side.
3. **Convergence threshold** — median $|\Delta\log_{10}\xi|$ vs $N_{\rm subvol}$.
4. **Full sweep** — all redshifts × mass cuts.

**Data:** `scripts/2pcf/slurm/submit_scope_xi_campaign.sh`  
**Reference (n=1024):** Corrfunc full-box run (`submit_corrfunc_xi_fullbox.sh`)  
**r range:** 0.01 → 271 Mpc/$h$ (30 log-spaced bins)  
**Redshifts:** iz155 ($z\approx1.5$), iz207 ($z\approx0.5$), iz271 ($z=0$)  
**Cuts:** no halo mass cut, no centrals filter; stellar mass cuts: none / 9 / 10 / 11

In [ ]:
import sys
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path('../../src').resolve()))
from galform_analysis.config import get_snapshot_redshift

try:
    from galform_analysis.utils.matplotlib_config import setconfig
    setconfig()
except ImportError:
    pass

In [ ]:
DATA_ROOT = Path('../../data/2pcf/scope_xi')
MODEL     = 'lc16'
SIM       = 'L800'
N_REF     = 1024   # full-box Corrfunc reference (pending — falls back to max available n)

# Quick-look config (sections 1–2)
QK_IZ        = 155
QK_MSTAR_TAG = 'mstar9.0'
QK_Z         = get_snapshot_redshift(f'iz{QK_IZ}')

# n values to show in detail and headline plots
PLOT_N     = [2,4,8,16,32,64, 128, 256]#[2, 8, 32, 128, 256]
HEADLINE_N = [2,4,8,16,32,64, 128, 256]#[4, 16, 64, 256]
# n=128 and n=256 are infeasible for mstar_none: runtime >> cosma8 8h/16h limits
PLOT_N_MSTAR_NONE = [n for n in PLOT_N if n < 128]

print(f'Data root:   {DATA_ROOT}')
print(f'Quick-look:  iz{QK_IZ}  →  z = {QK_Z:.3f}  ({QK_MSTAR_TAG})')

## Helper functions

In [ ]:
def load_and_aggregate(iz: int, mstar_tag: str) -> pd.DataFrame | None:
    """Load all scope_xi CSVs for one (iz, mstar_tag) and aggregate over seeds.

    The n=1024 Corrfunc full-box run is always used as the reference.  Residual
    columns (frac_diff_*, dlog_*) will be NaN until that reference is available.

    Returns a DataFrame with per-(n_subvol, bin_idx) statistics, or None if no data.
    """
    base = DATA_ROOT / MODEL / f'iz{iz}' / mstar_tag
    if not base.is_dir():
        return None

    files = sorted(base.glob(f'n*/seed*/scope_xi_{SIM}_iz{iz}.csv'))
    if not files:
        return None

    df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    df = df.drop_duplicates(subset=['selection_seed', 'n_subvol', 'bin_idx'])

    if N_REF not in df['n_subvol'].values:
        print(f'  WARNING: n={N_REF} reference not yet available for iz{iz} {mstar_tag} — residuals will be empty')

    # Compute statistics *only* over strictly positive ξ values when forming spreads/counts.
    # This avoids including zero or negative ξ in error/spread calculations.
    g = (df
         .groupby(['n_subvol', 'bin_idx', 'r_mid'], as_index=False)
         .agg(
             xi_corr_mean   = ('xi_corrected', lambda x: next((val for val in x if val != 0), np.nan)) ,#('xi_corrected', 'mean'),
             xi_corr_std    = ('xi_corrected', lambda x: float(np.nanstd(x[x > 0])) if np.any(x > 0) else np.nan),
             xi_naive_mean  = ('xi_naive',lambda x: next((val for val in x if val != 0), np.nan)), #('xi_naive',     'mean'),
             xi_naive_std   = ('xi_naive',     lambda x: float(np.nanstd(x[x > 0])) if np.any(x > 0) else np.nan),
             xi_corr_p16    = ('xi_corrected', lambda x: float(np.nanpercentile(x[x > 0], 16)) if np.any(x > 0) else np.nan),
             xi_corr_p84    = ('xi_corrected', lambda x: float(np.nanpercentile(x[x > 0], 84)) if np.any(x > 0) else np.nan),
             n_seeds_corr   = ('xi_corrected', lambda x: int(np.count_nonzero(x > 0))),
             n_seeds_naive  = ('xi_naive',     lambda x: int(np.count_nonzero(x > 0))),
         ))

    # Error on the mean using only the positive-value counts; leave NaN where no positive samples.
    g['xi_corr_err']  = np.where(g['n_seeds_corr'] > 0, g['xi_corr_std']  / np.sqrt(g['n_seeds_corr']),  np.nan)
    g['xi_naive_err'] = np.where(g['n_seeds_naive'] > 0, g['xi_naive_std'] / np.sqrt(g['n_seeds_naive']), np.nan)

    # Merge reference values using bin_idx (not r_mid, which can differ by rounding)
    ref = (g[g['n_subvol'] == N_REF][['bin_idx', 'xi_corr_mean', 'r_mid']]
             .rename(columns={'xi_corr_mean': 'xi_ref', 'r_mid': 'r_mid_ref'}))
    g = g.merge(ref, on='bin_idx', how='left')
    # Use reference r_mid (more accurate since it's from the full box)
    g['r_mid'] = g['r_mid_ref'].fillna(g['r_mid'])
    g = g.drop(columns=['r_mid_ref'])

    # Fractional residual: (xi - xi_ref) / |xi_ref|
    safe_ref = np.where(np.abs(g['xi_ref']) > 0, g['xi_ref'], np.nan)
    g['frac_diff_corr']  = (g['xi_corr_mean']  - g['xi_ref']) / np.abs(safe_ref)
    g['frac_diff_naive'] = (g['xi_naive_mean'] - g['xi_ref']) / np.abs(safe_ref)

    # log-ratio where both are positive (for threshold plot)
    pos_c = (g['xi_corr_mean'] > 0) & (g['xi_ref'] > 0)
    g['dlog_corr']  = np.where(pos_c, np.log10(g['xi_corr_mean'] / g['xi_ref']),  np.nan)
    pos_n = (g['xi_naive_mean'] > 0) & (g['xi_ref'] > 0)
    g['dlog_naive'] = np.where(pos_n, np.log10(g['xi_naive_mean'] / g['xi_ref']), np.nan)

    return g


def _n_colors(n_values):
    cmap = mpl.colormaps['plasma']
    return [cmap(0.1 + 0.75 * i / max(len(n_values) - 1, 1)) for i in range(len(n_values))]


def _mstar_label(mstar_tag: str) -> str:
    if mstar_tag == 'mstar_none':
        return 'no $M_*$ cut'
    val = mstar_tag.replace('mstar', '')
    return f'$M_*/M_\\odot h^{{-1}} > {val}$'


print('Helpers defined.')

In [ ]:
def four_panel(g: pd.DataFrame, iz: int, z: float, mstar_tag: str,
               plot_n_sel: list[int]) -> None:
    """4-panel: (naïve, SCOPE corrected) × (ξ(r), fractional residual)."""
    avail = sorted(g['n_subvol'].unique())
    pn    = [n for n in plot_n_sel if n in avail and n != N_REF]
    if not pn:
        print(f'  No requested n values available for iz{iz} {mstar_tag}')
        return
    
    # Check if reference exists
    if N_REF not in avail:
        print(f'  Skipping residual plots for iz{iz} {mstar_tag} (n=1024 reference not available)')
        return
    
    colors = _n_colors(pn)

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    (ax_tl, ax_tr), (ax_bl, ax_br) = axes

    ref = g[g['n_subvol'] == N_REF].sort_values('r_mid')
    ref_pos = ref[ref['xi_corr_mean'] > 0]
    if not ref_pos.empty:
        for ax in [ax_tl, ax_tr]:
            ax.plot(ref_pos['r_mid'], ref_pos['xi_corr_mean'],
                    color='black', lw=2.5, zorder=10,
                    label=f'$N_{{\\rm subvol}}={N_REF}$ (Corrfunc full box)')
    for ax in [ax_bl, ax_br]:
        ax.axhline(0, color='black', lw=1.5, ls='--', zorder=10)
        for lev in [0.05, 0.1]:
            ax.axhspan(-lev, lev, color='grey', alpha=0.07, zorder=0)

    for n, col in zip(pn, colors):
        sub = g[g['n_subvol'] == n].sort_values('r_mid')
        kw  = dict(color=col, ms=4, lw=1.2, elinewidth=0.7, capsize=2,
                   label=f'$N_{{\\rm subvol}}={n}$')

        pos_n = sub['xi_naive_mean'] > 0
        pos_c = sub['xi_corr_mean']  > 0
        ax_tl.errorbar(sub.loc[pos_n, 'r_mid'], sub.loc[pos_n, 'xi_naive_mean'],
                       yerr=sub.loc[pos_n, 'xi_naive_err'], fmt='o', **kw)
        ax_tr.errorbar(sub.loc[pos_c, 'r_mid'], sub.loc[pos_c, 'xi_corr_mean'],
                       yerr=sub.loc[pos_c, 'xi_corr_err'],  fmt='o', **kw)

        kw_b = dict(fmt='o', color=col, ms=4, lw=1.2, elinewidth=0.7, capsize=2)
        fin_n = sub['frac_diff_naive'].notna()
        fin_c = sub['frac_diff_corr'].notna()
        ax_bl.errorbar(sub.loc[fin_n, 'r_mid'], sub.loc[fin_n, 'frac_diff_naive'],
                       yerr=(sub.loc[fin_n, 'xi_naive_err'] /
                             np.abs(sub.loc[fin_n, 'xi_ref'])), **kw_b)
        ax_br.errorbar(sub.loc[fin_c, 'r_mid'], sub.loc[fin_c, 'frac_diff_corr'],
                       yerr=(sub.loc[fin_c, 'xi_corr_err'] /
                             np.abs(sub.loc[fin_c, 'xi_ref'])), **kw_b)

    for ax in [ax_tl, ax_tr]:
        ax.set_xscale('log'); ax.set_yscale('log')
        ax.set_ylabel(r'$\xi(r)$', fontsize=12)
        ax.legend(ncol=2, fontsize=9)
    for ax in [ax_bl, ax_br]:
        ax.set_xscale('log')
        ax.set_xlabel(r'$r$ [$h^{-1}$Mpc]', fontsize=12)
        ax.set_ylabel(r'$(\xi - \xi_{\rm ref})/|\xi_{\rm ref}|$', fontsize=12)
        ax.set_ylim(-1, 1)

    ax_tl.set_title('Density correction', fontsize=12)
    ax_tr.set_title('SCOPE corrected', fontsize=12)
    ax_bl.set_title('Density correction residual vs full box', fontsize=12)
    ax_br.set_title('SCOPE residual vs full box', fontsize=12)

    handles = [mpl.lines.Line2D([0],[0], color=c, lw=2, marker='o', ms=5,
                                label=f'$N_{{\\rm subvol}}={n}$')
               for n, c in zip(pn, colors)]
    ax_bl.legend(handles=handles, ncol=2, fontsize=9)
    ax_br.legend(handles=handles, ncol=2, fontsize=9)

    fig.suptitle(
        f'$\\xi(r)$ convergence — {SIM}/{MODEL}  iz{iz} ($z={z:.2f}$)  '
        f'{_mstar_label(mstar_tag)}',
        fontsize=14,
    )
    plt.tight_layout()
    plt.show()


def headline_comparison(iz: int, z: float, mstar_tag: str,
                         n_show: list[int] | None = None) -> None:
    """Single figure: SCOPE corrected with seed-spread shading plus naïve overlay."""
    g = load_and_aggregate(iz, mstar_tag)
    if g is None:
        print(f'No data: iz{iz} {mstar_tag}')
        return
    avail  = sorted(g['n_subvol'].unique())
    
    # Check if reference exists
    if N_REF not in avail:
        print(f'Skipping residual plots for iz{iz} {mstar_tag} (n=1024 reference not available)')
        return
    
    n_show = [n for n in (n_show or HEADLINE_N) if n in avail and n != N_REF]
    if not n_show:
        print(f'No data for requested n values: iz{iz} {mstar_tag}')
        return
    colors = _n_colors(n_show)

    fig, (ax_top, ax_bot) = plt.subplots(2, 1, figsize=(10, 10), sharex=True)

    ref = g[g['n_subvol'] == N_REF].sort_values('r_mid')
    ref_pos = ref[ref['xi_corr_mean'] > 0]
    if not ref_pos.empty:
        ax_top.plot(ref_pos['r_mid'], ref_pos['xi_corr_mean'],
                    color='black', lw=2.5, zorder=10,
                    label=f'$N_{{\\rm subvol}}={N_REF}$ (full box)')
    ax_bot.axhline(0, color='black', lw=1.5, ls='--', zorder=10)
    for lev in [0.05, 0.1]:
        ax_bot.axhspan(-lev, lev, color='grey', alpha=0.07, zorder=0)

    for n, col in zip(n_show, colors):
        sub = g[g['n_subvol'] == n].sort_values('r_mid')
        pos_c = sub['xi_corr_mean'] > 0

        ax_top.fill_between(sub.loc[pos_c, 'r_mid'],
                             sub.loc[pos_c, 'xi_corr_p16'],
                             sub.loc[pos_c, 'xi_corr_p84'],
                             color=col, alpha=0.2)
        ax_top.plot(sub.loc[pos_c, 'r_mid'], sub.loc[pos_c, 'xi_corr_mean'],
                    color=col, lw=1.8, label=f'SCOPE  $N={n}$')

        pos_n = sub['xi_naive_mean'] > 0
        ax_top.plot(sub.loc[pos_n, 'r_mid'], sub.loc[pos_n, 'xi_naive_mean'],
                    color=col, lw=1.3, ls='--', alpha=0.65, label=f'Naïve  $N={n}$')

        fin_c = sub['frac_diff_corr'].notna()
        ax_bot.plot(sub.loc[fin_c, 'r_mid'], sub.loc[fin_c, 'frac_diff_corr'],
                    color=col, lw=1.8)
        fin_n = sub['frac_diff_naive'].notna()
        ax_bot.plot(sub.loc[fin_n, 'r_mid'], sub.loc[fin_n, 'frac_diff_naive'],
                    color=col, lw=1.3, ls='--', alpha=0.65)

    ax_top.set_xscale('log'); ax_top.set_yscale('log')
    ax_top.set_ylabel(r'$\xi(r)$', fontsize=13)
    ax_top.legend(ncol=2, fontsize=9)
    ax_top.set_title(
        f'SCOPE vs naïve — {SIM}/{MODEL}  iz{iz} ($z={z:.2f}$)  '
        f'{_mstar_label(mstar_tag)}',
        fontsize=13,
    )

    ax_bot.set_xscale('log')
    ax_bot.set_xlabel(r'$r$ [$h^{-1}$Mpc]', fontsize=13)
    ax_bot.set_ylabel(r'$(\xi - \xi_{\rm ref})/|\xi_{\rm ref}|$', fontsize=13)
    ax_bot.set_ylim(-0.5, 0.5)
    ax_bot.set_title('Fractional residual  (solid = SCOPE, dashed = naïve)', fontsize=12)

    plt.tight_layout()
    plt.show()


print('Plot helpers defined.')

In [ ]:
def convergence_threshold(iz_list: list[int], mstar_tags: list[str]) -> None:
    """Median |Δlog₁₀ξ| vs N_subvol across r bins where ξ > 0."""
    cfg_colors  = mpl.colormaps['tab10'](np.linspace(0, 0.7, max(len(mstar_tags), 1)))
    iz_markers  = dict(zip(iz_list, ['o', 's', '^', 'D', 'v']))

    fig, (ax_l, ax_r) = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
    ax_l.set_title('SCOPE corrected', fontsize=12)
    ax_r.set_title('Naïve (no correction)', fontsize=12)

    for thresh in [0.01, 0.05, 0.1]:
        for ax in [ax_l, ax_r]:
            ax.axhline(thresh, color='lightgrey', lw=1.0, ls=':', zorder=0)
            ax.text(1.02, thresh, f'{thresh:.0%}', transform=ax.get_yaxis_transform(),
                    fontsize=8, va='center', color='grey')

    has_data = False
    for mstar_tag, cfg_col in zip(mstar_tags, cfg_colors):
        for iz in iz_list:
            z = get_snapshot_redshift(f'iz{iz}')
            g = load_and_aggregate(iz, mstar_tag)
            if g is None:
                continue
            ns = sorted(n for n in g['n_subvol'].unique() if n != N_REF)
            if not ns:
                continue
            med_c = [np.nanmedian(np.abs(g.loc[g['n_subvol']==n, 'dlog_corr' ].dropna())) for n in ns]
            med_n = [np.nanmedian(np.abs(g.loc[g['n_subvol']==n, 'dlog_naive'].dropna())) for n in ns]
            # Skip if all NaN
            if not np.any(np.isfinite(med_c)) and not np.any(np.isfinite(med_n)):
                continue
            has_data = True
            label = f'{_mstar_label(mstar_tag)}  $z={z:.1f}$'
            mkr   = iz_markers.get(iz, 'o')
            ax_l.plot(ns, med_c, marker=mkr, color=cfg_col, lw=1.8, ms=6, label=label)
            ax_r.plot(ns, med_n, marker=mkr, color=cfg_col, lw=1.8, ms=6, label=label)

    if not has_data:
        print('No finite convergence data yet (n=1024 reference not available)')
        return

    for ax in [ax_l, ax_r]:
        ax.set_xscale('log'); ax.set_yscale('log')
        ax.set_xlabel(r'$N_{\rm subvol}$', fontsize=12)
        ax.set_ylabel(r'median $|\Delta\log_{10}\xi|$', fontsize=12)
        ax.legend(fontsize=8, loc='upper right')

    fig.suptitle(f'$\\xi(r)$ convergence threshold — {SIM}/{MODEL}', fontsize=14)
    plt.tight_layout()
    plt.show()


def compute_convergence_metrics(g: pd.DataFrame) -> pd.DataFrame:
    """Compute multiple convergence metrics per N_subvol.

    Returns one row per n_subvol (excluding N_REF) with columns:
      med_dlog_{corr,naive}   — median |Δlog₁₀ξ|
      rms_frac_{corr,naive}   — RMS fractional error
      max_frac_{corr,naive}   — worst-case |frac error| over r bins
      f5_{corr,naive}         — fraction of r bins within 5% of reference
      f10_{corr,naive}        — fraction of r bins within 10% of reference
      med_scatter             — median seed scatter: (p84−p16)/(2|ξ_ref|)
    All metrics computed only over bins where ξ_ref > 0.
    """
    records = []
    for n in sorted(g['n_subvol'].unique()):
        if n == N_REF:
            continue
        sub  = g[(g['n_subvol'] == n) & (g['xi_ref'] > 0)].copy()
        if sub.empty:
            continue
        fc = sub['frac_diff_corr'].dropna().values
        fn = sub['frac_diff_naive'].dropna().values
        dc = sub['dlog_corr'].dropna().values
        dn = sub['dlog_naive'].dropna().values
        scatter = ((sub['xi_corr_p84'] - sub['xi_corr_p16']) /
                   (2 * sub['xi_ref'].abs())).dropna().values
        records.append({
            'n_subvol':       int(n),
            'med_dlog_corr':  np.nanmedian(np.abs(dc))                   if len(dc) else np.nan,
            'med_dlog_naive': np.nanmedian(np.abs(dn))                   if len(dn) else np.nan,
            'rms_frac_corr':  np.sqrt(np.nanmean(fc**2))                 if len(fc) else np.nan,
            'rms_frac_naive': np.sqrt(np.nanmean(fn**2))                 if len(fn) else np.nan,
            'max_frac_corr':  float(np.nanmax(np.abs(fc)))               if len(fc) else np.nan,
            'max_frac_naive': float(np.nanmax(np.abs(fn)))               if len(fn) else np.nan,
            'f5_corr':        float(np.mean(np.abs(fc) < 0.05))          if len(fc) else np.nan,
            'f5_naive':       float(np.mean(np.abs(fn) < 0.05))          if len(fn) else np.nan,
            'f10_corr':       float(np.mean(np.abs(fc) < 0.10))          if len(fc) else np.nan,
            'f10_naive':      float(np.mean(np.abs(fn) < 0.10))          if len(fn) else np.nan,
            'med_scatter':    float(np.nanmedian(scatter))                if len(scatter) else np.nan,
        })
    return pd.DataFrame(records)


def convergence_heatmap(iz: int, mstar_tag: str) -> None:
    """Heatmap of |frac residual| in (N_subvol, r) space for SCOPE corrected."""
    g = load_and_aggregate(iz, mstar_tag)
    if g is None:
        print(f'No data: iz{iz} {mstar_tag}')
        return
    
    # Check if we have finite residual data (requires n=1024 reference)
    n_finite = g['frac_diff_corr'].notna().sum()
    if n_finite == 0:
        print(f'No finite residuals: iz{iz} {mstar_tag} (n=1024 reference not yet available)')
        return
    
    z   = get_snapshot_redshift(f'iz{iz}')
    sub = g[g['n_subvol'] != N_REF].copy()
    ns  = sorted(sub['n_subvol'].unique())
    rs  = sorted(sub['r_mid'].unique())

    mat = np.full((len(ns), len(rs)), np.nan)
    for i, n in enumerate(ns):
        for j, r in enumerate(rs):
            v = sub.loc[(sub['n_subvol']==n) & (np.isclose(sub['r_mid'], r)),
                        'frac_diff_corr']
            if not v.empty:
                mat[i, j] = float(np.abs(v.iloc[0]))

    fig, ax = plt.subplots(figsize=(12, 5))
    im = ax.pcolormesh(
        rs, np.arange(len(ns)),
        np.clip(mat, 1e-4, None),
        cmap='RdYlGn_r',
        vmin=1e-4,
        vmax=1,
    )
    ax.set_xscale('log')
    ax.set_yticks(np.arange(len(ns)))
    ax.set_yticklabels([str(n) for n in ns])
    ax.set_xlabel(r'$r\,[h^{-1}$Mpc$]$', fontsize=12)
    ax.set_ylabel(r'$N_{\rm subvol}$', fontsize=12)
    cb = fig.colorbar(im, ax=ax, label=r'$|\Delta\xi/\xi_{\rm ref}|$')
    cb.set_ticks([0.01, 0.05, 0.1, 0.5, 1.0])
    cb.set_ticklabels(['1%', '5%', '10%', '50%', '100%'])
    ax.set_title(
        f'{SIM}/{MODEL} ($z={z:.2f}$)  '
        f'{_mstar_label(mstar_tag)}',
        fontsize=13,
    )
    plt.tight_layout()
    plt.show()


print('Threshold, metrics, and heatmap helpers defined.')

## 0  Data availability

In [ ]:
IZ_LIST     = [155, 207, 271]
MSTAR_TAGS  = ['mstar_none', 'mstar9.0', 'mstar10.0', 'mstar11.0']

print(f'{'iz':>6}  {'mstar_tag':>12}  {'n values available':>40}  seeds (max)')
print('-' * 80)
for iz in IZ_LIST:
    for mtag in MSTAR_TAGS:
        base = DATA_ROOT / MODEL / f'iz{iz}' / mtag
        files = sorted(base.glob(f'n*/seed*/scope_xi_{SIM}_iz{iz}.csv')) if base.is_dir() else []
        if not files:
            print(f'{iz:>6}  {mtag:>12}  (no data yet)')
            continue
        tmp = pd.concat([pd.read_csv(f, nrows=1) for f in files], ignore_index=True)
        ns  = sorted(tmp['n_subvol'].unique())
        max_seeds = tmp.groupby('n_subvol')['selection_seed'].nunique().max()
        print(f'{iz:>6}  {mtag:>12}  {str(ns):>40}  {max_seeds}')

## 1  Quick-look (single config)

In [ ]:
g = load_and_aggregate(QK_IZ, QK_MSTAR_TAG)
if g is None:
    print(f'No data yet for iz{QK_IZ} {QK_MSTAR_TAG}')
else:
    print(f'n values available: {sorted(g["n_subvol"].unique())}')
    four_panel(g, QK_IZ, QK_Z, QK_MSTAR_TAG, PLOT_N)

In [ ]:
g = load_and_aggregate(QK_IZ, QK_MSTAR_TAG)
if g is None:
    print(f'No data yet for iz{QK_IZ} {QK_MSTAR_TAG}')
else:
    print(f'n values available: {sorted(g["n_subvol"].unique())}')
    avail = sorted(g["n_subvol"].unique())
    pn = [n for n in avail if n != N_REF]
    if not pn:
        print(f'No requested n values available for iz{QK_IZ} {QK_MSTAR_TAG}')
    else:
        colors = _n_colors(pn)
        fig, (ax_top, ax_mid, ax_bot) = plt.subplots(3, 1, figsize=(7.5, 10.5), sharex=True)

        # Full-box reference (if available)
        if N_REF in avail:
            ref = g[g["n_subvol"] == N_REF].sort_values("r_mid")
            ref_pos = ref[ref["xi_corr_mean"] > 0]
            if not ref_pos.empty:
                ax_top.plot(
                    ref_pos["r_mid"],
                    ref_pos["xi_corr_mean"],
                    color="black",
                    lw=2.2,
                    zorder=10,
                    label=f"$N_{{\\rm subvol}}={N_REF}$ (full box)",
                )

        y_vals = []
        y_errs = []
        p_vals = []
        p_errs = []
        for n, col in zip(pn, colors):
            sub = g[g["n_subvol"] == n].sort_values("r_mid")
            pos_n = sub["xi_naive_mean"] > 0
            ax_top.errorbar(
                sub.loc[pos_n, "r_mid"],
                sub.loc[pos_n, "xi_naive_mean"],
                yerr=sub.loc[pos_n, "xi_naive_err"],
                fmt="o",
                color=col,
                ms=4,
                lw=1.2,
                elinewidth=0.7,
                capsize=2,
                label=f"$N_{{\\rm subvol}}={n}$",
            )

            fin_n = sub["frac_diff_naive"].notna()
            if fin_n.any():
                y = np.abs(sub.loc[fin_n, "frac_diff_naive"].to_numpy())
                y_err = (
                    sub.loc[fin_n, "xi_naive_err"] / sub.loc[fin_n, "xi_ref"].abs()
                ).to_numpy()
                p = 100.0 * y
                p_err = 100.0 * y_err
                y_vals.append(y)
                y_errs.append(y_err)
                p_vals.append(p)
                p_errs.append(p_err)
                ax_mid.errorbar(
                    sub.loc[fin_n, "r_mid"],
                    y,
                    yerr=y_err,
                    fmt="o",
                    color=col,
                    ms=4,
                    lw=1.2,
                    elinewidth=0.7,
                    capsize=2,
                )
                ax_bot.errorbar(
                    sub.loc[fin_n, "r_mid"],
                    p,
                    yerr=p_err,
                    fmt="o",
                    color=col,
                    ms=4,
                    lw=1.2,
                    elinewidth=0.7,
                    capsize=2,
                )

        ax_top.set_xscale("log")
        ax_top.set_yscale("log")
        ax_top.set_ylabel(r"$\xi(r)$", fontsize=12)
        ax_top.set_title(
            f"Sub-volume density estimation - {SIM}/{MODEL}  iz{QK_IZ} (z={QK_Z:.2f})  "
            f"{_mstar_label(QK_MSTAR_TAG)}",
            fontsize=12,
        )
        ax_top.legend(ncol=2, fontsize=9)

        def _abs_limits(vals, errs, min_abs):
            if not vals:
                return min_abs, 10.0 * min_abs
            y_all = np.concatenate(vals)
            y_err_all = np.concatenate(errs) if errs else np.zeros_like(y_all)
            y_hi = y_all + y_err_all
            y_max = float(np.nanmax(y_hi))
            if not np.isfinite(y_max):
                return min_abs, 10.0 * min_abs
            y_max = max(y_max, min_abs)
            y_min = max(min_abs, y_max / 1e4)
            return 1e-1*y_min, 1e1 * y_max

        ax_mid.set_xscale("log")
        ax_mid.set_yscale("log")
        ax_mid.set_ylabel(r"$|(\xi - \xi_{\rm ref})/\xi_{\rm ref}|$", fontsize=12)
        y_min, y_max = _abs_limits(y_vals, y_errs, min_abs=1e-3)
        ax_mid.set_ylim(1e-2, y_max*1e1)
        ax_mid.set_title("Absolute residual vs full box", fontsize=11)

        ax_bot.set_xscale("log")
        ax_bot.set_yscale("log")
        ax_bot.set_xlabel(r"$r$ [$h^{-1}$Mpc]", fontsize=12)
        ax_bot.set_ylabel(r"$|\Delta\xi/\xi_{\rm ref}|$ [\%]", fontsize=12)
        p_min, p_max = _abs_limits(p_vals, p_errs, min_abs=1e-1)
        ax_bot.set_ylim(1e-4, p_max*1e1)
        ax_bot.set_title("Absolute percent difference vs full box", fontsize=11)

        plt.tight_layout()
        plt.show()

## 2  Headline: SCOPE vs naïve with seed-spread bands

In [ ]:
headline_comparison(QK_IZ, QK_Z, QK_MSTAR_TAG, n_show=HEADLINE_N)

## 3  Convergence threshold

Median $|\Delta\log_{10}\xi|$ vs $N_{\rm subvol}$ across all $r$ bins where $\xi > 0$.  
Left panel shows SCOPE converges; right panel shows naïve diverges at small $N$.

In [ ]:
convergence_threshold(IZ_LIST, MSTAR_TAGS)

## 3b  Multi-metric convergence analysis

Four complementary metrics, all vs $N_{\rm subvol}$, broken out by stellar mass selection and redshift.
Solid lines = SCOPE corrected; dashed = naïve.

| Metric | What it captures |
|--------|-----------------|
| RMS fractional error | Overall bias across all $r$ bins |
| Max \|frac error\| | Worst-case single bin — most sensitive to outliers |
| Fraction of bins within 5% | Practical convergence fraction |
| Median seed scatter | Statistical noise: $(p_{84}-p_{16})/(2|\xi_{\rm ref}|)$, independent of reference |

In [ ]:
def multi_metric_convergence(iz_list: list[int], mstar_tags: list[str]) -> None:
    """2×2 grid of convergence metrics vs N_subvol.

    Each panel shows one metric.  Lines coloured by mstar selection, markers by iz.
    Solid = SCOPE corrected, dashed = naïve where applicable.
    """
    # (column_key, y-label, naive_key, log_y, reference_thresholds)
    panels = [
        ('rms_frac_corr',  r'RMS fractional error',                 'rms_frac_naive',  True,  [0.05, 0.10]),
        ('max_frac_corr',  r'Max $|\Delta\xi/\xi_{\rm ref}|$',      'max_frac_naive',  True,  [0.05, 0.10, 0.20]),
        ('f5_corr',        r'Fraction of $r$ bins within 5\%',      'f5_naive',        False, [0.5, 0.8, 1.0]),
        ('med_scatter',    r'Median seed scatter $(p_{84}-p_{16})/2|\xi_{\rm ref}|$',
                           None, True, [0.05, 0.10]),
    ]

    colors  = mpl.colormaps['tab10'](np.linspace(0, 0.7, max(len(mstar_tags), 1)))
    markers = {155: 'o', 207: 's', 271: '^'}

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    for ax, (met_key, ylabel, naive_key, log_y, thresholds) in zip(axes.flatten(), panels):
        for mstar_tag, col in zip(mstar_tags, colors):
            for iz in iz_list:
                g = load_and_aggregate(iz, mstar_tag)
                if g is None:
                    continue
                m = compute_convergence_metrics(g)
                if m.empty or met_key not in m.columns:
                    continue
                valid = m[m[met_key].notna()]
                if valid.empty:
                    continue
                z   = get_snapshot_redshift(f'iz{iz}')
                mkr = markers.get(iz, 'o')
                lbl = f'{_mstar_label(mstar_tag)}  $z={z:.1f}$'
                ax.plot(valid['n_subvol'], valid[met_key],
                        color=col, marker=mkr, ms=6, lw=1.8, label=lbl)
                if naive_key and naive_key in m.columns:
                    vn = m[m[naive_key].notna()]
                    if not vn.empty:
                        ax.plot(vn['n_subvol'], vn[naive_key],
                                color=col, marker=mkr, ms=4, lw=1.0, ls='--', alpha=0.45)

        # Reference lines
        for thresh in thresholds:
            ax.axhline(thresh, color='lightgrey', lw=0.9, ls=':', zorder=0)
            ax.text(1.02, thresh, f'{thresh:.0%}', transform=ax.get_yaxis_transform(),
                    fontsize=8, va='center', color='grey')

        ax.set_xscale('log')
        if log_y:
            ax.set_yscale('log')
        ax.set_xlabel(r'$N_{\rm subvol}$', fontsize=11)
        ax.set_ylabel(ylabel, fontsize=11)

    # Shared legends
    col_handles = [mpl.lines.Line2D([0],[0], color=c, lw=2,
                   label=_mstar_label(t)) for t, c in zip(mstar_tags, colors)]
    iz_handles  = [mpl.lines.Line2D([0],[0], color='k', marker=markers[iz],
                   ms=7, lw=0, label=f'iz{iz} ($z={get_snapshot_redshift(f"iz{iz}"):.1f}$)')
                   for iz in iz_list]
    style_handles = [
        mpl.lines.Line2D([0],[0], color='k', lw=1.8,       label='SCOPE corrected'),
        mpl.lines.Line2D([0],[0], color='k', lw=1.0, ls='--', alpha=0.55, label='Naïve'),
    ]
    axes[0, 0].legend(handles=col_handles,   fontsize=8, loc='upper right', title='Selection')
    axes[0, 1].legend(handles=iz_handles,    fontsize=8, loc='upper right', title='Redshift')
    axes[1, 1].legend(handles=style_handles, fontsize=9, loc='lower right')

    fig.suptitle(
        f'Convergence metrics vs $N_{{\\rm subvol}}$ — {SIM}/{MODEL}',
        fontsize=14,
    )
    plt.tight_layout()
    plt.show()


def scale_resolved_convergence(iz: int, mstar_tag: str,
                                r_targets: list[float] | None = None) -> None:
    """Fractional residual at specific r scales vs N_subvol.

    Shows whether convergence is scale-dependent: 1-halo, transition, 2-halo.
    Top panel = SCOPE corrected, bottom = naïve.
    """
    g = load_and_aggregate(iz, mstar_tag)
    if g is None:
        print(f'No data: iz{iz} {mstar_tag}')
        return
    z      = get_snapshot_redshift(f'iz{iz}')
    all_r  = sorted(g['r_mid'].unique())
    if r_targets is None:
        r_targets = [0.3, 3.0, 30.0]   # 1-halo, transition, 2-halo

    # Snap each target to nearest bin in log space
    sel_r = []
    for rt in r_targets:
        closest = min(all_r, key=lambda r: abs(np.log10(r) - np.log10(rt)))
        if closest not in sel_r:
            sel_r.append(closest)

    ns      = sorted(n for n in g['n_subvol'].unique() if n != N_REF)
    palette = mpl.colormaps['plasma'](np.linspace(0.15, 0.85, len(sel_r)))

    fig, (ax_top, ax_bot) = plt.subplots(2, 1, figsize=(9, 9), sharex=True)
    for ax in (ax_top, ax_bot):
        ax.axhline(0, color='k', lw=1.2, ls='--', zorder=5)
        for lev in [0.05, 0.10]:
            ax.axhspan(-lev, lev, color='grey', alpha=0.06, zorder=0)

    for r_sel, col in zip(sel_r, palette):
        sub_r  = g[np.isclose(g['r_mid'], r_sel)]
        n_vals, fc_vals, fn_vals = [], [], []
        for n in ns:
            row = sub_r[sub_r['n_subvol'] == n]
            if row.empty:
                continue
            fc = float(row['frac_diff_corr'].iloc[0])
            fn = float(row['frac_diff_naive'].iloc[0])
            if np.isfinite(fc):
                n_vals.append(n); fc_vals.append(fc); fn_vals.append(fn)
        if not n_vals:
            continue
        lbl = f'$r \\approx {r_sel:.2g}$ $h^{{-1}}$Mpc'
        ax_top.plot(n_vals, fc_vals, color=col, marker='o', ms=6, lw=1.8, label=lbl)
        ax_bot.plot(n_vals, fn_vals, color=col, marker='o', ms=5, lw=1.3)

    for ax in (ax_top, ax_bot):
        ax.set_xscale('log')
        ax.set_ylabel(r'$(\xi - \xi_{\rm ref})/|\xi_{\rm ref}|$', fontsize=12)
        ax.set_ylim(-0.6, 0.6)
    ax_bot.set_xlabel(r'$N_{\rm subvol}$', fontsize=12)
    ax_top.set_title('SCOPE corrected', fontsize=12)
    ax_bot.set_title('Naïve', fontsize=12)
    ax_top.legend(fontsize=10, title='Scale')

    fig.suptitle(
        f'Scale-resolved convergence — {SIM}/{MODEL}  iz{iz} ($z={z:.2f}$)  '
        f'{_mstar_label(mstar_tag)}',
        fontsize=13,
    )
    plt.tight_layout()
    plt.show()


print('multi_metric_convergence and scale_resolved_convergence defined.')

In [ ]:
# Multi-metric: all selections × all redshifts
multi_metric_convergence(IZ_LIST, MSTAR_TAGS)

# Scale-resolved: quick-look config — do all mstar cuts to compare how scale-dependence shifts
for mtag in MSTAR_TAGS:
    scale_resolved_convergence(QK_IZ, mtag)

## 4  Convergence heatmap

$|\xi_{\rm SCOPE}/\xi_{\rm ref} - 1|$ as a function of $N_{\rm subvol}$ and $r$.
Green = <1% error, red = >50% error.

In [ ]:
convergence_heatmap(QK_IZ, QK_MSTAR_TAG)

## 5  Full sweep — all redshifts × mass cuts

In [ ]:
for iz in IZ_LIST:
    z = get_snapshot_redshift(f'iz{iz}')
    print(f'\n{"="*60}\niz{iz}  z = {z:.3f}\n{"="*60}')
    for mtag in MSTAR_TAGS:
        g = load_and_aggregate(iz, mtag)
        if g is None:
            print(f'  No data yet: {mtag} — skipping')
            continue
        n_avail = sorted(g['n_subvol'].unique())
        print(f'  {mtag} — n_subvol: {n_avail}')
        four_panel(g, iz, z, mtag, PLOT_N_MSTAR_NONE if mtag == 'mstar_none' else PLOT_N)
        convergence_heatmap(iz, mtag)

In [ ]:
# Galaxy-count heatmap
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.ticker import LogFormatterMathtext, LogLocator
import numpy as np
import pandas as pd

counts_csv = Path("../../data/2pcf/scope_xi_ngal_counts.csv")
if not counts_csv.is_file():
    raise FileNotFoundError(f"Missing CSV: {counts_csv}")

def _mstar_label(tag: str) -> str:
    if tag == "mstar_none":
        return r"no $M_*$ cut"
    val = tag.replace("mstar", "")
    return rf"$\log_{{10}}(M_*/M_\odot\,h^{{-1}}) > {val}$"

# Darker = higher N_gal
cmap = plt.get_cmap("magma_r").copy()
cmap.set_bad("#f0f0f0")

df = pd.read_csv(counts_csv)
df = df.dropna(subset=["ngal", "n_subvol", "iz", "mstar_tag"])
df["ngal"] = df["ngal"].astype(int)
df["n_subvol"] = df["n_subvol"].astype(int)
df["iz"] = df["iz"].astype(int)

mstar_order = ["mstar9.0", "mstar10.0", "mstar11.0"] # ["mstar_none", "mstar9.0", "mstar10.0", "mstar11.0"]
mstar_tags = [t for t in mstar_order if t in df["mstar_tag"].unique()]
if not mstar_tags:
    mstar_tags = sorted(df["mstar_tag"].unique())

iz_list = sorted(df["iz"].unique())
n_subvols = sorted(df["n_subvol"].unique())

agg = (
    df.groupby(["iz", "mstar_tag", "n_subvol"], as_index=False)
    .agg(ngal_mean=("ngal", "mean"))
)

heat = np.full((len(iz_list), len(mstar_tags), len(n_subvols)), np.nan, dtype=float)
for i, iz in enumerate(iz_list):
    for j, mstar_tag in enumerate(mstar_tags):
        sub = agg[(agg["iz"] == iz) & (agg["mstar_tag"] == mstar_tag)]
        for _, row in sub.iterrows():
            k = n_subvols.index(int(row["n_subvol"]))
            heat[i, j, k] = float(row["ngal_mean"])

finite = np.isfinite(heat) & (heat > 0)
if not np.any(finite):
    raise RuntimeError("No finite ngal values available for plotting.")

vmin = float(np.nanmin(heat[finite]))
vmax = float(np.nanmax(heat[finite]))

ncols = max(len(iz_list), 1)
fig, axes = plt.subplots(
    nrows=1,
    ncols=ncols,
    figsize=(4.8 * ncols, 3.8),
    sharey=True,
    constrained_layout=True,
)
if ncols == 1:
    axes = np.array([axes])

for i, iz in enumerate(iz_list):
    ax = axes[i]
    masked = np.ma.masked_invalid(heat[i])
    masked = np.ma.masked_less_equal(masked, 0)
    im = ax.imshow(
        masked,
        aspect="auto",
        origin="upper",
        norm=LogNorm(vmin=vmin, vmax=vmax),
        cmap=cmap,
    )
    z = get_snapshot_redshift(f"iz{iz}")
    ax.set_title(f"z = {z:.2f}")
    ax.set_xticks(np.arange(len(n_subvols)))
    ax.set_xticklabels([str(n) for n in n_subvols], rotation=45, ha="right")
    ax.set_yticks(np.arange(len(mstar_tags)))
    ax.set_yticklabels([_mstar_label(t) for t in mstar_tags])
    ax.set_xlabel(r"$N_{\rm subvol}$")

fig.suptitle("Galaxy counts per selection")
cbar = fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.9, pad=0.02)
cbar.locator = LogLocator(base=10)
cbar.formatter = LogFormatterMathtext(base=10)
cbar.update_ticks()
cbar.set_label(r"$N_{\rm gal}$")
plt.show()